## 案例: 演示 集成学习之 Bagging思想 随机森林算法 代码.

集成学习:

    概述:
        把多个弱学习器 组成 1个强学习器的过程 -> 集成学习.
    思想:
        Bagging思想:
            1. 有放回的随机抽样.
            2. 平权投票.
            3. 可以并行执行.

        Boosting思想:
            1. 每次训练都会使用全部样本.
            2. 加权投票 -> 预测正确:权重降低, 预测错误: 权重增加.
            3. 只能串行执行.
            
    Bagging思想代表:
        随机森林算法.

随机森林算法:

    1. 每个弱学习器都是 CART树(必须是二叉树)
    2. 有放回的随机抽样, 平权投票, 并行执行.

#### 导包

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split        # 切分训练集和测试集
from sklearn.tree import DecisionTreeClassifier             # 决策树
from sklearn.ensemble import RandomForestClassifier         # 随机森林算法(分类器)
from sklearn.model_selection import GridSearchCV            # 网格搜索

#### 1. 加载数据

In [4]:
df = pd.read_csv('./data/train.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


#### 2. 数据的预处理

In [5]:
# 2.1 抽取特征和标签
x = df[['Pclass', 'Sex', 'Age']].copy()         # 船舱等级，性别，年龄
y = df['Survived']

# 2.2 空值处理，用Age列的平均值填充Age列的空值
x['Age'] = x['Age'].fillna(x['Age'].mean())

# 2.3 热编码处理
x = pd.get_dummies(x)

# 2.4 划分训练集和测试集
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=23)

#### 3. 特征工程（略）

#### 4. 模型训练，预测，评估

###### 场景1： 单一决策树

In [8]:
# 4.1 创建决策树对象，
estimator1 = DecisionTreeClassifier()
# 4.2 模型训练
estimator1.fit(x_train, y_train)
# 4.3 预测
y_pred = estimator1.predict(x_test)
print(f'预测值为：{y_pred}')
# 4.4 评估
print(f'决策树模型准确率为：{estimator1.score(x_test, y_test)}')

预测值为：[1 0 0 0 0 0 0 0 1 0 0 0 1 0 0 1 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 1 1 0 0 0 1 0 0 0 0 0 1 0 1 1 0 0 0 1 0 1 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0
 0 0 1 0 1 1 0 1 1 0 0 0 1 0 1 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 1 0
 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 1 1 0 0 1 0 0 0 1 1 0 0 0 1 1 1 1 1 0 0 0
 1 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 1 1 1]
决策树模型准确率为：0.8044692737430168


###### 场景2： 随机森林算法 -> 采用默认参数

In [13]:
# 4.1 创建随机森林对象，演示：多个决策树（Bagging思想）效果
estimator2 = RandomForestClassifier(random_state=23)       # n_estimators=100, max_depth=None
# 4.2 模型训练
estimator2.fit(x_train, y_train)
# 4.3 预测
y_pred2 = estimator2.predict(x_test)
print(f'预测值为：{y_pred2}')
# 4.4 评估
print(f'随机森林模型的准确率为：{estimator2.score(x_test, y_test)}')

预测值为：[1 0 0 1 0 0 0 0 1 0 0 0 1 0 0 1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 1 1 0 0 0 1 0 0 0 0 0 1 0 1 1 0 0 0 1 0 1 0 0 0 0 0 0 0 1 1 1 0 1 0 0 0 0
 0 0 1 0 1 1 0 1 1 0 0 0 1 0 1 0 1 0 1 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 1 1 0
 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 1 1 0 0 1 0 0 0 1 1 0 0 0 1 1 1 1 1 0 0 0
 1 0 1 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 1 1 1]
随机森林模型的准确率为：0.8100558659217877


###### 场景3： 随机森林算法 -> 采用网格搜索

In [16]:
# 4.1 创建随机森林对象，演示：多个决策树（Bagging思想）效果
# 新增random_state保证结果可复现（可选但推荐）
estimator3 = RandomForestClassifier(random_state=23) 
estimator3.fit(x_train, y_train)

# 4.2 参数准备
params = {'n_estimators': [30, 50, 60, 90, 110], 'max_depth':[2, 3, 5, 7]}

# 4.3 创建网格搜索对象，结合交叉验证
gs_estimator = GridSearchCV(estimator3, param_grid=params, cv=2)

# 4.4 模型训练
gs_estimator.fit(x_train, y_train)

# 4.5 预测
y_pred3 = gs_estimator.predict(x_test)
print(f'预测值为{y_pred3}')

# 4.6 评估（核心修改：用网格搜索后的最优模型评估）
# 错误：estimator3.score(x_test, y_test) → 正确：gs_estimator.score(x_test, y_test)
print(f'随机森林模型的准确率为：{gs_estimator.score(x_test, y_test)}')

# 4.7 获取最佳参数
print(f'最佳参数为：{gs_estimator.best_params_}')

预测值为[1 0 0 1 0 0 0 0 1 0 0 0 1 0 0 1 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 1 1 0 0 0 1 0 0 0 0 0 1 0 1 1 0 1 0 1 0 1 0 0 0 0 0 0 0 0 1 1 0 1 0 0 0 1
 0 0 1 0 1 1 0 1 0 0 0 0 1 0 1 0 0 0 1 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 1 0 0
 0 0 0 0 0 1 1 0 0 1 0 0 0 0 0 0 1 1 0 0 1 0 0 0 1 1 0 0 0 1 1 1 1 1 0 0 0
 1 0 1 0 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 1 1 1]
随机森林模型的准确率为：0.7988826815642458
最佳参数为：{'max_depth': 5, 'n_estimators': 50}
